# ✈️ 机场代码匹配逻辑详解

## 📋 学习目标
- 理解 ICAO 机场代码系统
- 掌握机场数据中不同代码字段的含义
- 学会处理真实世界数据的不一致性问题
- 熟练使用 pandas query 方法进行数据筛选

## 🎯 核心问题
**为什么在 `airports_to_parquet.py` 中需要同时检查 `gps_code` 和 `ident` 两个字段？**

```python
af = af.query("gps_code.isin(@usedairports) or ident.isin(@usedairports)")
```

# 1️⃣ ICAO 代码基础知识

## 🌍 什么是 ICAO？

**ICAO** = **International Civil Aviation Organization**（国际民航组织）

### 📋 ICAO 机场代码特点

- **长度**：4个字符（字母）
- **全球唯一**：每个机场都有唯一的ICAO代码
- **地区前缀**：通过前缀区分不同地区
- **官方标准**：航空业官方使用的标准代码

### 🏷️ 机场代码对比表

| 代码类型 | 长度 | 示例 | 用途 | 覆盖范围 |
|---------|------|------|------|----------|
| **ICAO** | 4个字母 | ZBAA, KJFK | 航空管制、飞行计划 | 全球所有机场 |
| **IATA** | 3个字母 | PEK, JFK | 商业航空、票务系统 | 主要商业机场 |
| **本地代码** | 不定 | K1G4, CN87 | 地方航空管理 | 各国内部使用 |

### 🌏 ICAO 代码地区前缀举例

- **Z** - 中国大陆（如：ZBAA北京首都、ZSSS上海虹桥）
- **K** - 美国本土（如：KJFK纽约肯尼迪、KLAX洛杉矶）
- **E** - 北欧、东欧（如：EGLL伦敦希思罗）
- **V** - 印度、东南亚（如：VTBS曼谷素万那普）
- **R** - 日本、韩国（如：RJTT东京羽田、RKSI首尔仁川）

# 2️⃣ 导入必要的库

In [2]:
# 导入数据处理库
import pandas as pd
import numpy as np

# 设置 pandas 显示选项，让输出更清晰
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

print("✅ 库导入成功！")

✅ 库导入成功！


# 3️⃣ 创建演示机场数据

模拟 `ourairports2024-10-21.csv` 中的机场数据结构，展示不同类型机场的代码特征。

In [3]:
# 创建模拟的机场数据
airport_data = {
    'ident': [
        'ZBAA',      # 北京首都机场 - ICAO代码
        'PEK',       # 北京首都机场 - IATA代码
        'ZSSS',      # 上海虹桥机场 - ICAO代码
        'SHA',       # 上海虹桥机场 - IATA代码
        'KORD',      # 芝加哥奥黑尔机场 - ICAO代码
        'ORD',       # 芝加哥奥黑尔机场 - IATA代码
        'EGLL',      # 伦敦希思罗机场 - ICAO代码
        'LHR',       # 伦敦希思罗机场 - IATA代码
        'CN-0001',   # 小型地方机场 - 本地代码
        'US-SMALL-1' # 小型私人机场 - 本地代码
    ],
    'name': [
        'Beijing Capital International Airport',
        'Beijing Capital International Airport',
        'Shanghai Hongqiao International Airport',
        'Shanghai Hongqiao International Airport',
        'Chicago O\'Hare International Airport',
        'Chicago O\'Hare International Airport',
        'London Heathrow Airport',
        'London Heathrow Airport',
        'Small Local Airport China',
        'Small Private Airport US'
    ],
    'type': [
        'large_airport',
        'large_airport',
        'large_airport',
        'large_airport',
        'large_airport',
        'large_airport',
        'large_airport',
        'large_airport',
        'small_airport',
        'small_airport'
    ],
    'gps_code': [
        'ZBAA',      # GPS代码通常与ICAO相同
        None,        # IATA条目没有GPS代码
        'ZSSS',
        None,
        'KORD',
        None,
        'EGLL',
        None,
        'CN-0001',   # 小机场可能有GPS代码
        None         # 有些小机场没有GPS代码
    ],
    'iata_code': [
        'PEK',       # ICAO条目关联的IATA代码
        'PEK',       # IATA条目的IATA代码
        'SHA',
        'SHA',
        'ORD',
        'ORD',
        'LHR',
        'LHR',
        None,        # 小机场可能没有IATA代码
        None
    ]
}

# 转换为DataFrame
af = pd.DataFrame(airport_data)
print("📊 模拟机场数据表：")
print(af.head(10))

📊 模拟机场数据表：
        ident                           name           type gps_code iata_code
0        ZBAA  Beijing Capital Internatio...  large_airport     ZBAA       PEK
1         PEK  Beijing Capital Internatio...  large_airport     None       PEK
2        ZSSS  Shanghai Hongqiao Internat...  large_airport     ZSSS       SHA
3         SHA  Shanghai Hongqiao Internat...  large_airport     None       SHA
4        KORD  Chicago O'Hare Internation...  large_airport     KORD       ORD
5         ORD  Chicago O'Hare Internation...  large_airport     None       ORD
6        EGLL        London Heathrow Airport  large_airport     EGLL       LHR
7         LHR        London Heathrow Airport  large_airport     None       LHR
8     CN-0001      Small Local Airport China  small_airport  CN-0001      None
9  US-SMALL-1       Small Private Airport US  small_airport     None      None


# 4️⃣ 创建"实际使用的机场"列表

这个列表模拟从飞行轨迹数据中提取出的机场代码。

In [4]:
# 模拟从飞行轨迹数据中提取的机场代码
# 在实际项目中，这些代码来自于飞行数据的origin/destination字段

used_airports = {
    'ZBAA',      # 北京首都机场 - ICAO格式
    'SHA',       # 上海虹桥机场 - IATA格式  
    'ORD',       # 芝加哥奥黑尔机场 - IATA格式
    'EGLL',      # 伦敦希思罗机场 - ICAO格式
    'CN-0001'    # 小型机场 - 本地代码格式
}

print("✈️ 飞行轨迹中实际使用的机场代码：")
print(f"机场数量: {len(used_airports)}")
print(f"机场代码: {sorted(used_airports)}")

# 注意：这个集合包含了不同格式的机场代码
# - ICAO代码：ZBAA, EGLL
# - IATA代码：SHA, ORD  
# - 本地代码：CN-0001
print("\n🔍 代码格式分析：")
for code in sorted(used_airports):
    if len(code) == 4 and code.isalpha():
        print(f"  {code} - 可能是ICAO代码（4个字母）")
    elif len(code) == 3 and code.isalpha():
        print(f"  {code} - 可能是IATA代码（3个字母）")
    else:
        print(f"  {code} - 本地或特殊格式代码")

✈️ 飞行轨迹中实际使用的机场代码：
机场数量: 5
机场代码: ['CN-0001', 'EGLL', 'ORD', 'SHA', 'ZBAA']

🔍 代码格式分析：
  CN-0001 - 本地或特殊格式代码
  EGLL - 可能是ICAO代码（4个字母）
  ORD - 可能是IATA代码（3个字母）
  SHA - 可能是IATA代码（3个字母）
  ZBAA - 可能是ICAO代码（4个字母）


# 5️⃣ 核心匹配逻辑：pandas查询语法详解

现在我们来分析关键代码：`af.query("gps_code.isin(@used_airports) or ident.isin(@used_airports)")`

In [5]:
# 🔍 逐步解析查询语法

print("🎯 查询语法分解：")
print("完整查询：af.query(\"gps_code.isin(@used_airports) or ident.isin(@used_airports)\")")
print()

# 第一部分：gps_code.isin(@used_airports)
print("📍 第一部分：gps_code.isin(@used_airports)")
print("   - gps_code：DataFrame的GPS代码列")
print("   - .isin()：pandas方法，检查值是否在指定集合中")
print("   - @used_airports：查询语法中的变量引用，指向我们的used_airports集合")
print()

# 第二部分：ident.isin(@used_airports)  
print("🆔 第二部分：ident.isin(@used_airports)")
print("   - ident：DataFrame的标识符代码列")
print("   - .isin()：同样的包含检查方法")
print("   - @used_airports：同样的机场代码集合")
print()

# 逻辑连接：or
print("🔗 逻辑连接：or")
print("   - 满足任一条件即可：GPS代码匹配 OR 标识符代码匹配")
print("   - 这样可以捕获所有相关的机场记录")
print()

# 实际执行查询
print("🚀 执行查询：")
filtered_airports = af.query("gps_code.isin(@used_airports) or ident.isin(@used_airports)")
print(f"匹配到的机场记录数：{len(filtered_airports)}")
print()
print("匹配结果：")

🎯 查询语法分解：
完整查询：af.query("gps_code.isin(@used_airports) or ident.isin(@used_airports)")

📍 第一部分：gps_code.isin(@used_airports)
   - gps_code：DataFrame的GPS代码列
   - .isin()：pandas方法，检查值是否在指定集合中
   - @used_airports：查询语法中的变量引用，指向我们的used_airports集合

🆔 第二部分：ident.isin(@used_airports)
   - ident：DataFrame的标识符代码列
   - .isin()：同样的包含检查方法
   - @used_airports：同样的机场代码集合

🔗 逻辑连接：or
   - 满足任一条件即可：GPS代码匹配 OR 标识符代码匹配
   - 这样可以捕获所有相关的机场记录

🚀 执行查询：
匹配到的机场记录数：5

匹配结果：


In [6]:
# 显示匹配结果
print(filtered_airports)
print()

# 🔍 详细分析每个匹配
print("🔍 匹配详情分析：")
for idx, row in filtered_airports.iterrows():
    ident_match = row['ident'] in used_airports
    gps_match = row['gps_code'] in used_airports if pd.notna(row['gps_code']) else False
    
    print(f"✅ {row['name']}:")
    print(f"   - ident: {row['ident']} {'✓' if ident_match else '✗'}")
    print(f"   - gps_code: {row['gps_code']} {'✓' if gps_match else '✗'}")
    print(f"   - 匹配原因: {'ident匹配' if ident_match else ''}{'gps_code匹配' if gps_match else ''}")
    print()

     ident                           name           type gps_code iata_code
0     ZBAA  Beijing Capital Internatio...  large_airport     ZBAA       PEK
3      SHA  Shanghai Hongqiao Internat...  large_airport     None       SHA
5      ORD  Chicago O'Hare Internation...  large_airport     None       ORD
6     EGLL        London Heathrow Airport  large_airport     EGLL       LHR
8  CN-0001      Small Local Airport China  small_airport  CN-0001      None

🔍 匹配详情分析：
✅ Beijing Capital International Airport:
   - ident: ZBAA ✓
   - gps_code: ZBAA ✓
   - 匹配原因: ident匹配gps_code匹配

✅ Shanghai Hongqiao International Airport:
   - ident: SHA ✓
   - gps_code: None ✗
   - 匹配原因: ident匹配

✅ Chicago O'Hare International Airport:
   - ident: ORD ✓
   - gps_code: None ✗
   - 匹配原因: ident匹配

✅ London Heathrow Airport:
   - ident: EGLL ✓
   - gps_code: EGLL ✓
   - 匹配原因: ident匹配gps_code匹配

✅ Small Local Airport China:
   - ident: CN-0001 ✓
   - gps_code: CN-0001 ✓
   - 匹配原因: ident匹配gps_code匹配



# 6️⃣ 等价的传统写法对比

让我们看看这个查询语句的等价写法，帮助理解。

In [7]:
# 方法1：使用query()方法（原始代码的写法）
method1 = af.query("gps_code.isin(@used_airports) or ident.isin(@used_airports)")
print("📝 方法1 - query()方法：")
print("af.query(\"gps_code.isin(@used_airports) or ident.isin(@used_airports)\")")
print(f"结果行数：{len(method1)}")
print()

# 方法2：使用布尔索引（传统写法）
condition1 = af['gps_code'].isin(used_airports)
condition2 = af['ident'].isin(used_airports)
method2 = af[condition1 | condition2]
print("📝 方法2 - 布尔索引：")
print("condition1 = af['gps_code'].isin(used_airports)")
print("condition2 = af['ident'].isin(used_airports)")
print("af[condition1 | condition2]")
print(f"结果行数：{len(method2)}")
print()

# 方法3：分步骤执行（最详细的写法）
gps_matches = af['gps_code'].isin(used_airports)
ident_matches = af['ident'].isin(used_airports)
combined_condition = gps_matches | ident_matches
method3 = af[combined_condition]
print("📝 方法3 - 分步执行：")
print("gps_matches = af['gps_code'].isin(used_airports)")
print("ident_matches = af['ident'].isin(used_airports)")
print("combined_condition = gps_matches | ident_matches")
print("af[combined_condition]")
print(f"结果行数：{len(method3)}")
print()

# 验证三种方法结果相同
print("🔍 验证结果：")
print(f"方法1与方法2结果相同：{method1.equals(method2)}")
print(f"方法2与方法3结果相同：{method2.equals(method3)}")
print(f"所有方法结果相同：{method1.equals(method2) and method2.equals(method3)}")

📝 方法1 - query()方法：
af.query("gps_code.isin(@used_airports) or ident.isin(@used_airports)")
结果行数：5

📝 方法2 - 布尔索引：
condition1 = af['gps_code'].isin(used_airports)
condition2 = af['ident'].isin(used_airports)
af[condition1 | condition2]
结果行数：5

📝 方法3 - 分步执行：
gps_matches = af['gps_code'].isin(used_airports)
ident_matches = af['ident'].isin(used_airports)
combined_condition = gps_matches | ident_matches
af[combined_condition]
结果行数：5

🔍 验证结果：
方法1与方法2结果相同：True
方法2与方法3结果相同：True
所有方法结果相同：True


# 7️⃣ 为什么需要双重匹配？

解释为什么要同时检查 `gps_code` 和 `ident` 字段。

In [8]:
# 🤔 为什么需要检查两个字段？

print("💡 双重匹配的原因：")
print()

# 原因1：数据不一致性
print("1️⃣ 数据不一致性：")
print("   - 不同数据源使用不同的机场代码格式")
print("   - 飞行数据可能混合使用ICAO、IATA、本地代码")
print("   - 机场数据库需要覆盖所有可能的匹配方式")
print()

# 原因2：字段用途不同
print("2️⃣ 字段用途差异：")
print("   - ident：机场的主要标识符（可能是ICAO、IATA或本地代码）")
print("   - gps_code：GPS/导航系统使用的代码（通常是ICAO格式）")
print("   - 同一机场可能在不同字段有不同代码")
print()

# 原因3：确保完整性
print("3️⃣ 确保匹配完整性：")
print("   - 避免遗漏：某些机场可能只在一个字段中匹配")
print("   - 最大化覆盖：提高找到正确机场的概率")
print()

# 实际案例演示
print("📊 实际案例分析：")
for airport_code in used_airports:
    ident_matches = af[af['ident'] == airport_code]
    gps_matches = af[af['gps_code'] == airport_code]
    
    print(f"🔍 查找代码 '{airport_code}'：")
    print(f"   - ident字段匹配：{len(ident_matches)}条记录")
    print(f"   - gps_code字段匹配：{len(gps_matches)}条记录")
    
    if len(ident_matches) > 0:
        print(f"   - ident匹配示例：{ident_matches.iloc[0]['name']}")
    if len(gps_matches) > 0:
        print(f"   - gps_code匹配示例：{gps_matches.iloc[0]['name']}")
    print()

💡 双重匹配的原因：

1️⃣ 数据不一致性：
   - 不同数据源使用不同的机场代码格式
   - 飞行数据可能混合使用ICAO、IATA、本地代码
   - 机场数据库需要覆盖所有可能的匹配方式

2️⃣ 字段用途差异：
   - ident：机场的主要标识符（可能是ICAO、IATA或本地代码）
   - gps_code：GPS/导航系统使用的代码（通常是ICAO格式）
   - 同一机场可能在不同字段有不同代码

3️⃣ 确保匹配完整性：
   - 避免遗漏：某些机场可能只在一个字段中匹配
   - 最大化覆盖：提高找到正确机场的概率

📊 实际案例分析：
🔍 查找代码 'EGLL'：
   - ident字段匹配：1条记录
   - gps_code字段匹配：1条记录
   - ident匹配示例：London Heathrow Airport
   - gps_code匹配示例：London Heathrow Airport

🔍 查找代码 'ORD'：
   - ident字段匹配：1条记录
   - gps_code字段匹配：0条记录
   - ident匹配示例：Chicago O'Hare International Airport

🔍 查找代码 'SHA'：
   - ident字段匹配：1条记录
   - gps_code字段匹配：0条记录
   - ident匹配示例：Shanghai Hongqiao International Airport

🔍 查找代码 'ZBAA'：
   - ident字段匹配：1条记录
   - gps_code字段匹配：1条记录
   - ident匹配示例：Beijing Capital International Airport
   - gps_code匹配示例：Beijing Capital International Airport

🔍 查找代码 'CN-0001'：
   - ident字段匹配：1条记录
   - gps_code字段匹配：1条记录
   - ident匹配示例：Small Local Airport China
   - gps_code匹配示例：Small Local Airport China



# 8️⃣ 总结与最佳实践

关键要点和实际应用建议。

In [9]:
# 🎯 总结：机场代码匹配逻辑

print("📚 核心概念总结：")
print()

print("🏷️ 机场代码类型：")
print("   - ICAO：国际民航组织4字母代码（如ZBAA、EGLL）")
print("   - IATA：国际航空运输协会3字母代码（如PEK、LHR）")  
print("   - GPS代码：导航系统代码（通常与ICAO相同）")
print("   - 本地代码：特定地区的标识符")
print()

print("🔍 查询语法要点：")
print("   - .query()：pandas的字符串查询方法")
print("   - .isin()：检查值是否在集合中")
print("   - @变量名：在查询字符串中引用外部变量")
print("   - or：逻辑或运算符")
print()

print("💡 设计原则：")
print("   - 多字段匹配：增加找到正确机场的概率")
print("   - 容错性：处理不同数据源的格式差异")
print("   - 完整性：确保不遗漏任何相关机场")
print()

print("🛠️ 实际应用建议：")
print("   1. 了解数据源的代码格式特点")
print("   2. 根据需要调整匹配字段")
print("   3. 验证匹配结果的准确性")
print("   4. 考虑数据清理和标准化")
print()

print("✅ 学习收获：")
print("   - 掌握了pandas query语法")
print("   - 理解了机场代码系统")
print("   - 学会了数据匹配策略")
print("   - 认识了实际数据处理挑战")

📚 核心概念总结：

🏷️ 机场代码类型：
   - ICAO：国际民航组织4字母代码（如ZBAA、EGLL）
   - IATA：国际航空运输协会3字母代码（如PEK、LHR）
   - GPS代码：导航系统代码（通常与ICAO相同）
   - 本地代码：特定地区的标识符

🔍 查询语法要点：
   - .query()：pandas的字符串查询方法
   - .isin()：检查值是否在集合中
   - @变量名：在查询字符串中引用外部变量
   - or：逻辑或运算符

💡 设计原则：
   - 多字段匹配：增加找到正确机场的概率
   - 容错性：处理不同数据源的格式差异
   - 完整性：确保不遗漏任何相关机场

🛠️ 实际应用建议：
   1. 了解数据源的代码格式特点
   2. 根据需要调整匹配字段
   3. 验证匹配结果的准确性
   4. 考虑数据清理和标准化

✅ 学习收获：
   - 掌握了pandas query语法
   - 理解了机场代码系统
   - 学会了数据匹配策略
   - 认识了实际数据处理挑战


# 🔥 额外讲解：pandas.query() 语法详解

既然你第一次见这个语法，让我们深入讲解一下！

In [7]:
# 🎯 pandas.query() 语法从零开始讲解

print("🔥 pandas.query() 语法详解")
print("=" * 40)
print()

# 1. 最基本的语法结构
print("1️⃣ 基本结构：")
print("   DataFrame.query('查询条件字符串')")
print("   注意：条件是字符串格式！")
print()

# 2. 简单示例
simple_data = pd.DataFrame({
    'name': ['机场A', '机场B', '机场C'],
    'size': [100, 200, 50],
    'type': ['大型', '大型', '小型']
})

print("📊 简单示例数据：")
print(simple_data)
print()

print("🔍 简单查询：")
print("代码：simple_data.query('size > 100')")
result = simple_data.query('size > 100')
print("结果：")
print(result)
print()

🔥 pandas.query() 语法详解

1️⃣ 基本结构：
   DataFrame.query('查询条件字符串')
   注意：条件是字符串格式！

📊 简单示例数据：
  name  size type
0  机场A   100   大型
1  机场B   200   大型
2  机场C    50   小型

🔍 简单查询：
代码：simple_data.query('size > 100')
结果：
  name  size type
1  机场B   200   大型



In [8]:
# 3. 关键概念：.isin() 方法
print("2️⃣ 关键概念：.isin() 方法")
print("   作用：检查某个值是否在指定的列表/集合中")
print()

# 创建示例
codes_data = pd.DataFrame({
    'airport_code': ['ZBAA', 'ZSSS', 'ZGGG', 'ZGSZ'],
    'city': ['北京', '上海', '广州', '深圳']
})

my_target_codes = ['ZBAA', 'ZSSS']  # 我关心的机场代码

print("📊 机场代码数据：")
print(codes_data)
print()
print(f"🎯 目标代码列表：{my_target_codes}")
print()

# 传统方法
print("🔸 传统方法：")
print("codes_data[codes_data['airport_code'].isin(my_target_codes)]")
traditional = codes_data[codes_data['airport_code'].isin(my_target_codes)]
print(traditional)
print()

# query方法
print("🔸 query方法：")
print("codes_data.query('airport_code.isin(@my_target_codes)')")
query_method = codes_data.query('airport_code.isin(@my_target_codes)')
print(query_method)
print()

2️⃣ 关键概念：.isin() 方法
   作用：检查某个值是否在指定的列表/集合中

📊 机场代码数据：
  airport_code city
0         ZBAA   北京
1         ZSSS   上海
2         ZGGG   广州
3         ZGSZ   深圳

🎯 目标代码列表：['ZBAA', 'ZSSS']

🔸 传统方法：
codes_data[codes_data['airport_code'].isin(my_target_codes)]
  airport_code city
0         ZBAA   北京
1         ZSSS   上海

🔸 query方法：
codes_data.query('airport_code.isin(@my_target_codes)')
  airport_code city
0         ZBAA   北京
1         ZSSS   上海



In [9]:
# 4. 神奇的 @ 符号
print("3️⃣ 神奇的 @ 符号")
print("   @ 的作用：在query字符串中引用Python变量")
print()

my_list = ['ZBAA', 'ZSSS']

print("💡 对比理解：")
print("   - 没有@：query('airport_code.isin(my_list)')  ❌ 报错！")
print("   - 有了@：query('airport_code.isin(@my_list)') ✅ 正确！")
print()

# 演示错误情况
print("🚫 错误示例（会报错）：")
try:
    wrong = codes_data.query('airport_code.isin(my_list)')
except Exception as e:
    print(f"   错误信息：{e}")
print()

print("✅ 正确示例：")
print("codes_data.query('airport_code.isin(@my_list)')")
correct = codes_data.query('airport_code.isin(@my_list)')
print(correct)
print()

print("🧠 理解要点：")
print("   - query()内部是字符串，不能直接访问Python变量")
print("   - @符号告诉pandas：'这是外部的Python变量'")
print("   - 没有@，pandas会认为是DataFrame的列名")

3️⃣ 神奇的 @ 符号
   @ 的作用：在query字符串中引用Python变量

💡 对比理解：
   - 没有@：query('airport_code.isin(my_list)')  ❌ 报错！
   - 有了@：query('airport_code.isin(@my_list)') ✅ 正确！

🚫 错误示例（会报错）：
   错误信息：name 'my_list' is not defined

✅ 正确示例：
codes_data.query('airport_code.isin(@my_list)')
  airport_code city
0         ZBAA   北京
1         ZSSS   上海

🧠 理解要点：
   - query()内部是字符串，不能直接访问Python变量
   - @符号告诉pandas：'这是外部的Python变量'
   - 没有@，pandas会认为是DataFrame的列名


In [10]:
# 5. 逻辑运算符：or 和 and
print("4️⃣ 逻辑运算符：or 和 and")
print()

# 创建更完整的示例数据
airport_demo = pd.DataFrame({
    'ident': ['ZBAA', 'PEK', 'ZSSS', 'SHA', 'ZGGG'],
    'gps_code': ['ZBAA', None, 'ZSSS', None, 'ZGGG'],
    'name': ['北京首都-ICAO', '北京首都-IATA', '上海虹桥-ICAO', '上海虹桥-IATA', '广州白云'],
    'size': ['大', '大', '大', '大', '中']
})

target_airports = {'ZBAA', 'SHA', 'ZGGG'}

print("📊 完整机场数据：")
print(airport_demo)
print()
print(f"🎯 目标机场：{target_airports}")
print()

# 单一条件
print("🔸 单一条件查询：")
print("airport_demo.query('ident.isin(@target_airports)')")
single = airport_demo.query('ident.isin(@target_airports)')
print(single)
print()

# or 条件 - 这就是你的代码！
print("🔸 OR条件查询（你的代码逻辑）：")
print("airport_demo.query('gps_code.isin(@target_airports) or ident.isin(@target_airports)')")
or_result = airport_demo.query('gps_code.isin(@target_airports) or ident.isin(@target_airports)')
print(or_result)
print()

# and 条件对比
print("🔸 AND条件查询（对比）：")
print("airport_demo.query('gps_code.isin(@target_airports) and size == \"大\"')")
and_result = airport_demo.query('gps_code.isin(@target_airports) and size == "大"')
print(and_result)
print()

print("💡 OR vs AND 区别：")
print("   - OR：满足任一条件即可（更宽松）")
print("   - AND：必须同时满足所有条件（更严格）")

4️⃣ 逻辑运算符：or 和 and

📊 完整机场数据：
  ident gps_code       name size
0  ZBAA     ZBAA  北京首都-ICAO    大
1   PEK     None  北京首都-IATA    大
2  ZSSS     ZSSS  上海虹桥-ICAO    大
3   SHA     None  上海虹桥-IATA    大
4  ZGGG     ZGGG       广州白云    中

🎯 目标机场：{'ZBAA', 'SHA', 'ZGGG'}

🔸 单一条件查询：
airport_demo.query('ident.isin(@target_airports)')
  ident gps_code       name size
0  ZBAA     ZBAA  北京首都-ICAO    大
3   SHA     None  上海虹桥-IATA    大
4  ZGGG     ZGGG       广州白云    中

🔸 OR条件查询（你的代码逻辑）：
airport_demo.query('gps_code.isin(@target_airports) or ident.isin(@target_airports)')
  ident gps_code       name size
0  ZBAA     ZBAA  北京首都-ICAO    大
3   SHA     None  上海虹桥-IATA    大
4  ZGGG     ZGGG       广州白云    中

🔸 AND条件查询（对比）：
airport_demo.query('gps_code.isin(@target_airports) and size == "大"')
  ident gps_code       name size
0  ZBAA     ZBAA  北京首都-ICAO    大

💡 OR vs AND 区别：
   - OR：满足任一条件即可（更宽松）
   - AND：必须同时满足所有条件（更严格）


In [11]:
# 6. 完整语法总结
print("5️⃣ 你的代码完整解析")
print("=" * 40)
print()

print("🎯 原始代码：")
print('af.query("gps_code.isin(@usedairports) or ident.isin(@usedairports)")')
print()

print("📝 逐个部分解析：")
print("┌─ af.query(...)          → 对DataFrame af 执行查询")
print("│")
print("├─ gps_code.isin(...)    → 检查gps_code列的值是否在集合中")
print("│   └─ @usedairports     → 引用外部变量usedairports")
print("│")
print("├─ or                    → 逻辑或运算符")
print("│")
print("└─ ident.isin(...)       → 检查ident列的值是否在集合中")
print("    └─ @usedairports     → 同样引用外部变量")
print()

print("🔄 等价的传统写法：")
print("condition1 = af['gps_code'].isin(usedairports)")
print("condition2 = af['ident'].isin(usedairports)")  
print("result = af[condition1 | condition2]")
print()

print("🌟 query语法的优势：")
print("   ✅ 更接近自然语言")
print("   ✅ 复杂条件时更简洁")
print("   ✅ 类似SQL的写法")
print()

print("⚠️ 注意事项：")
print("   📌 条件必须用引号包围（字符串）")
print("   📌 引用外部变量要用@")
print("   📌 列名直接写，不要加引号")
print("   📌 字符串值需要用引号：size == \"大\"")
print()

print("🎉 现在你完全掌握了这个语法！")

5️⃣ 你的代码完整解析

🎯 原始代码：
af.query("gps_code.isin(@usedairports) or ident.isin(@usedairports)")

📝 逐个部分解析：
┌─ af.query(...)          → 对DataFrame af 执行查询
│
├─ gps_code.isin(...)    → 检查gps_code列的值是否在集合中
│   └─ @usedairports     → 引用外部变量usedairports
│
├─ or                    → 逻辑或运算符
│
└─ ident.isin(...)       → 检查ident列的值是否在集合中
    └─ @usedairports     → 同样引用外部变量

🔄 等价的传统写法：
condition1 = af['gps_code'].isin(usedairports)
condition2 = af['ident'].isin(usedairports)
result = af[condition1 | condition2]

🌟 query语法的优势：
   ✅ 更接近自然语言
   ✅ 复杂条件时更简洁
   ✅ 类似SQL的写法

⚠️ 注意事项：
   📌 条件必须用引号包围（字符串）
   📌 引用外部变量要用@
   📌 列名直接写，不要加引号
   📌 字符串值需要用引号：size == "大"

🎉 现在你完全掌握了这个语法！


# 🚀 高级语法解析：列表推导式+条件表达式+zip函数

现在来解析这个看起来很复杂的一行代码！

In [12]:
# 🎯 原始代码分析
print("🚀 复杂的一行代码解析")
print("=" * 50)
print()

original_code = "icao_code = [i if i in usedairports else g for g, i in zip(af.gps_code.values, af.ident.values)]"
print("🎯 原始代码：")
print(original_code)
print()

print("📝 这一行代码包含了三个重要概念：")
print("1️⃣ 列表推导式 [... for ... in ...]")
print("2️⃣ 条件表达式 (三元运算符) value1 if condition else value2")
print("3️⃣ zip函数 - 将两个列表配对")
print()

# 创建模拟数据进行演示
demo_af = pd.DataFrame({
    'ident': ['ZBAA', 'PEK', 'ZSSS', 'SHA', 'ZGGG'],
    'gps_code': ['ZBAA', None, 'ZSSS', None, 'ZGGG'],
    'name': ['北京首都-ICAO', '北京首都-IATA', '上海虹桥-ICAO', '上海虹桥-IATA', '广州白云']
})

demo_usedairports = {'ZBAA', 'SHA', 'ZGGG'}  # 航班中实际使用的机场代码

print("📊 演示数据：")
print(demo_af[['ident', 'gps_code']])
print()
print(f"✈️ 使用的机场代码：{demo_usedairports}")
print()

🚀 复杂的一行代码解析

🎯 原始代码：
icao_code = [i if i in usedairports else g for g, i in zip(af.gps_code.values, af.ident.values)]

📝 这一行代码包含了三个重要概念：
1️⃣ 列表推导式 [... for ... in ...]
2️⃣ 条件表达式 (三元运算符) value1 if condition else value2
3️⃣ zip函数 - 将两个列表配对

📊 演示数据：
  ident gps_code
0  ZBAA     ZBAA
1   PEK     None
2  ZSSS     ZSSS
3   SHA     None
4  ZGGG     ZGGG

✈️ 使用的机场代码：{'ZBAA', 'SHA', 'ZGGG'}



In [13]:
# 第一步：理解 zip() 函数
print("🔗 第一步：理解 zip() 函数")
print("-" * 30)
print()

gps_values = demo_af.gps_code.values
ident_values = demo_af.ident.values

print("📋 gps_code.values:")
print(f"   {gps_values}")
print()
print("📋 ident.values:")
print(f"   {ident_values}")
print()

print("🔗 zip(gps_code.values, ident.values) 的效果：")
zipped = list(zip(gps_values, ident_values))
print("   将两个列表一一配对:")
for i, (g, i_val) in enumerate(zipped):
    print(f"   索引{i}: gps_code='{g}', ident='{i_val}'")
print()

print("💡 zip的作用：")
print("   - 将两个列表逐个元素配对")
print("   - 返回元组的迭代器")
print("   - 可以同时遍历多个列表")

🔗 第一步：理解 zip() 函数
------------------------------

📋 gps_code.values:
   ['ZBAA' None 'ZSSS' None 'ZGGG']

📋 ident.values:
   ['ZBAA' 'PEK' 'ZSSS' 'SHA' 'ZGGG']

🔗 zip(gps_code.values, ident.values) 的效果：
   将两个列表一一配对:
   索引0: gps_code='ZBAA', ident='ZBAA'
   索引1: gps_code='None', ident='PEK'
   索引2: gps_code='ZSSS', ident='ZSSS'
   索引3: gps_code='None', ident='SHA'
   索引4: gps_code='ZGGG', ident='ZGGG'

💡 zip的作用：
   - 将两个列表逐个元素配对
   - 返回元组的迭代器
   - 可以同时遍历多个列表


In [14]:
# 第二步：理解条件表达式 (三元运算符)
print("🤔 第二步：理解条件表达式")
print("-" * 30)
print()

print("📝 条件表达式格式：")
print("   value1 if condition else value2")
print()
print("🔍 在你的代码中：")
print("   i if i in usedairports else g")
print()
print("📖 含义：")
print("   - 如果 i (ident值) 在使用的机场列表中")
print("   - 就选择 i (ident值)")
print("   - 否则选择 g (gps_code值)")
print()

print("🎯 逐个测试每对数据：")
for i, (g, i_val) in enumerate(zipped):
    # 执行条件判断
    if i_val in demo_usedairports:
        result = i_val
        reason = f"选择ident='{i_val}' (因为在使用列表中)"
    else:
        result = g
        reason = f"选择gps_code='{g}' (因为ident='{i_val}'不在使用列表中)"
    
    print(f"   索引{i}: {reason} → 结果: '{result}'")

🤔 第二步：理解条件表达式
------------------------------

📝 条件表达式格式：
   value1 if condition else value2

🔍 在你的代码中：
   i if i in usedairports else g

📖 含义：
   - 如果 i (ident值) 在使用的机场列表中
   - 就选择 i (ident值)
   - 否则选择 g (gps_code值)

🎯 逐个测试每对数据：
   索引0: 选择ident='ZBAA' (因为在使用列表中) → 结果: 'ZBAA'
   索引1: 选择gps_code='None' (因为ident='PEK'不在使用列表中) → 结果: 'None'
   索引2: 选择gps_code='ZSSS' (因为ident='ZSSS'不在使用列表中) → 结果: 'ZSSS'
   索引3: 选择ident='SHA' (因为在使用列表中) → 结果: 'SHA'
   索引4: 选择ident='ZGGG' (因为在使用列表中) → 结果: 'ZGGG'


In [15]:
# 第三步：理解列表推导式
print("📋 第三步：理解列表推导式")
print("-" * 30)
print()

print("📝 列表推导式格式：")
print("   [表达式 for 变量 in 可迭代对象]")
print()
print("🔍 在你的代码中：")
print("   [i if i in usedairports else g for g, i in zip(...)]")
print()
print("📖 含义：")
print("   - 对于每一对 (g, i)")
print("   - 应用条件表达式 'i if i in usedairports else g'")
print("   - 收集所有结果到一个新列表中")
print()

# 使用原始代码执行
print("🚀 执行原始代码：")
icao_code_result = [i if i in demo_usedairports else g for g, i in zip(demo_af.gps_code.values, demo_af.ident.values)]
print(f"   结果: {icao_code_result}")
print()

# 等价的传统写法
print("🔄 等价的传统写法：")
print("traditional_result = []")
print("for g, i in zip(demo_af.gps_code.values, demo_af.ident.values):")
print("    if i in demo_usedairports:")
print("        traditional_result.append(i)")
print("    else:")
print("        traditional_result.append(g)")

traditional_result = []
for g, i in zip(demo_af.gps_code.values, demo_af.ident.values):
    if i in demo_usedairports:
        traditional_result.append(i)
    else:
        traditional_result.append(g)

print(f"   结果: {traditional_result}")
print()
print(f"🔍 两种方法结果相同: {icao_code_result == traditional_result}")

📋 第三步：理解列表推导式
------------------------------

📝 列表推导式格式：
   [表达式 for 变量 in 可迭代对象]

🔍 在你的代码中：
   [i if i in usedairports else g for g, i in zip(...)]

📖 含义：
   - 对于每一对 (g, i)
   - 应用条件表达式 'i if i in usedairports else g'
   - 收集所有结果到一个新列表中

🚀 执行原始代码：
   结果: ['ZBAA', None, 'ZSSS', 'SHA', 'ZGGG']

🔄 等价的传统写法：
traditional_result = []
for g, i in zip(demo_af.gps_code.values, demo_af.ident.values):
    if i in demo_usedairports:
        traditional_result.append(i)
    else:
        traditional_result.append(g)
   结果: ['ZBAA', None, 'ZSSS', 'SHA', 'ZGGG']

🔍 两种方法结果相同: True


In [16]:
# 为什么要这样做？业务逻辑解释
print("🎯 为什么要这样做？业务逻辑解释")
print("=" * 40)
print()

print("💡 目标：创建统一的机场代码字段")
print("   - 优先使用航班数据中实际出现的代码格式")
print("   - 确保数据一致性")
print()

print("📊 详细分析每个结果：")
for i, (original_ident, gps_code, final_code) in enumerate(zip(demo_af.ident.values, demo_af.gps_code.values, icao_code_result)):
    print(f"📍 索引{i}:")
    print(f"   原始ident: '{original_ident}'")
    print(f"   gps_code: '{gps_code}'")
    print(f"   在使用列表中: {original_ident in demo_usedairports}")
    print(f"   最终选择: '{final_code}'")
    
    if original_ident in demo_usedairports:
        print(f"   ✅ 选择ident ('{original_ident}') 因为它在航班数据中出现")
    else:
        print(f"   🔄 选择gps_code ('{gps_code}') 因为ident不在航班数据中")
    print()

print("🎉 总结：")
print("   这样处理后，每个机场都有一个统一的代码")
print("   且这个代码是航班数据中实际使用的格式")
print("   避免了代码格式不匹配的问题")

🎯 为什么要这样做？业务逻辑解释

💡 目标：创建统一的机场代码字段
   - 优先使用航班数据中实际出现的代码格式
   - 确保数据一致性

📊 详细分析每个结果：
📍 索引0:
   原始ident: 'ZBAA'
   gps_code: 'ZBAA'
   在使用列表中: True
   最终选择: 'ZBAA'
   ✅ 选择ident ('ZBAA') 因为它在航班数据中出现

📍 索引1:
   原始ident: 'PEK'
   gps_code: 'None'
   在使用列表中: False
   最终选择: 'None'
   🔄 选择gps_code ('None') 因为ident不在航班数据中

📍 索引2:
   原始ident: 'ZSSS'
   gps_code: 'ZSSS'
   在使用列表中: False
   最终选择: 'ZSSS'
   🔄 选择gps_code ('ZSSS') 因为ident不在航班数据中

📍 索引3:
   原始ident: 'SHA'
   gps_code: 'None'
   在使用列表中: True
   最终选择: 'SHA'
   ✅ 选择ident ('SHA') 因为它在航班数据中出现

📍 索引4:
   原始ident: 'ZGGG'
   gps_code: 'ZGGG'
   在使用列表中: True
   最终选择: 'ZGGG'
   ✅ 选择ident ('ZGGG') 因为它在航班数据中出现

🎉 总结：
   这样处理后，每个机场都有一个统一的代码
   且这个代码是航班数据中实际使用的格式
   避免了代码格式不匹配的问题


In [17]:
# 语法知识总结
print("📚 语法知识点总结")
print("=" * 30)
print()

print("1️⃣ zip() 函数：")
print("   作用：将多个序列逐个配对")
print("   语法：zip(seq1, seq2, ...)")
print("   示例：list(zip([1,2,3], ['a','b','c'])) → [(1,'a'), (2,'b'), (3,'c')]")
print()

print("2️⃣ 条件表达式（三元运算符）：")
print("   作用：根据条件选择不同的值")
print("   语法：value1 if condition else value2")
print("   示例：'大' if age >= 18 else '小'")
print()

print("3️⃣ 列表推导式：")
print("   作用：用简洁语法创建列表")
print("   语法：[表达式 for 变量 in 可迭代对象]")
print("   示例：[x*2 for x in [1,2,3]] → [2,4,6]")
print()

print("4️⃣ 组合使用的强大之处：")
print("   ✅ 代码简洁：一行搞定复杂逻辑")
print("   ✅ 性能优秀：内部优化过的操作")
print("   ✅ 可读性强：熟悉后一眼看懂")
print("   ✅ Pythonic：符合Python风格")
print()

print("🎓 学习建议：")
print("   1. 先理解基础语法")
print("   2. 逐步组合使用")
print("   3. 多练习实际场景")
print("   4. 对比传统写法理解优势")
print()

print("🎉 恭喜！你已经掌握了Python的高级语法组合！")

📚 语法知识点总结

1️⃣ zip() 函数：
   作用：将多个序列逐个配对
   语法：zip(seq1, seq2, ...)
   示例：list(zip([1,2,3], ['a','b','c'])) → [(1,'a'), (2,'b'), (3,'c')]

2️⃣ 条件表达式（三元运算符）：
   作用：根据条件选择不同的值
   语法：value1 if condition else value2
   示例：'大' if age >= 18 else '小'

3️⃣ 列表推导式：
   作用：用简洁语法创建列表
   语法：[表达式 for 变量 in 可迭代对象]
   示例：[x*2 for x in [1,2,3]] → [2,4,6]

4️⃣ 组合使用的强大之处：
   ✅ 代码简洁：一行搞定复杂逻辑
   ✅ 性能优秀：内部优化过的操作
   ✅ 可读性强：熟悉后一眼看懂
   ✅ Pythonic：符合Python风格

🎓 学习建议：
   1. 先理解基础语法
   2. 逐步组合使用
   3. 多练习实际场景
   4. 对比传统写法理解优势

🎉 恭喜！你已经掌握了Python的高级语法组合！
